In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from skfolio import Population, RiskMeasure
from skfolio.preprocessing import prices_to_returns
from skfolio.optimization import MeanRisk, ObjectiveFunction,RiskBudgeting
from skfolio.prior import BlackLitterman as BL
from pathlib import Path


In [3]:
def load_returns(folder_path, min_years=5):
    X = pd.DataFrame()

    for file_path in sorted(Path(folder_path).glob("*.csv")):

        df = pd.read_csv(file_path)
        df["date"] = pd.to_datetime(df["date"])

        history = (df["date"].max() - df["date"].min()).days / 365
        
        if history < min_years:
            continue

        df = df.set_index("date")
        df.columns = ["close"]

        returns = prices_to_returns(df)

        ticker = file_path.stem.split(".")[0]
        returns.columns = [ticker]

        if X.empty:
            X = returns
        else:
            X = X.join(returns, how="inner")

    return X

Для MOEX

In [4]:
X = load_returns("data/moex")

print(X.head())

                ABRD      AFKS      AFLT      AKRN      ALRS      AMEZ  \
date                                                                     
2021-04-29  0.000000 -0.009564 -0.000924  0.002936  0.000370  0.020758   
2021-04-30  0.000000 -0.011932 -0.010487 -0.027001  0.002860 -0.017794   
2021-05-04 -0.002475 -0.013875  0.016833  0.009696 -0.003405  0.022257   
2021-05-05  0.000000  0.020059  0.000613  0.007285  0.030743  0.002532   
2021-05-06 -0.007444 -0.001762  0.004289 -0.012163  0.014510  0.008586   

                APTK      AQUA      ARSA      ASSB  ...      VTBR      WTCM  \
date                                                ...                       
2021-04-29 -0.006003 -0.003333  0.000000 -0.005388  ... -0.007952  0.022365   
2021-04-30 -0.001582 -0.010034 -0.002179  0.011738  ...  0.030060 -0.019883   
2021-05-04 -0.005904 -0.006757  0.037040 -0.007140  ...  0.056420 -0.018252   
2021-05-05  0.000869  0.001701  0.012615  0.011685  ... -0.001842  0.039255   
2021-05

In [5]:
X_train, X_test = train_test_split(X, test_size=0.33, shuffle=False)


In [6]:
X_train

,ABRD,AFKS,AFLT,AKRN,ALRS,AMEZ,APTK,AQUA,ARSA,ASSB,...,VTBR,WTCM,WTCMP,YAKG,YKEN,YKENP,YRSB,YRSBP,ZILL,ZVEZ
date,,,,,,,,,,,,,,,,,,,,,
2021-04-29,0.000000,-0.009564,-0.000924,0.002936,0.000370,0.020758,-0.006003,-0.003333,0.000000,-0.005388,...,-0.007952,0.022365,0.013508,-0.020699,-0.024249,-0.013089,0.009091,0.000000,0.037241,-0.001185
2021-04-30,0.000000,-0.011932,-0.010487,-0.027001,0.002860,-0.017794,-0.001582,-0.010034,-0.002179,0.011738,...,0.030060,-0.019883,-0.013328,0.003523,0.031953,-0.007958,-0.009009,-0.010811,-0.007979,-0.005931
2021-05-04,-0.002475,-0.013875,0.016833,0.009696,-0.003405,0.022257,-0.005904,-0.006757,0.037040,-0.007140,...,0.056420,-0.018252,-0.015768,-0.025889,-0.011468,-0.009358,0.000000,0.000000,-0.008043,-0.004773
2021-05-05,0.000000,0.020059,0.000613,0.007285,0.030743,0.002532,0.000869,0.001701,0.012615,0.011685,...,-0.001842,0.039255,0.029744,0.040090,0.017401,-0.004049,0.000000,0.000000,0.005405,0.005995
2021-05-06,-0.007444,-0.001762,0.004289,-0.012163,0.014510,0.008586,-0.000289,-0.008489,0.035250,0.001777,...,0.001845,-0.005967,0.000000,0.001732,-0.001140,-0.009485,0.018182,0.005464,-0.005376,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-09,-0.000980,-0.015358,0.006381,0.003949,0.000558,0.021914,0.005442,-0.008123,-0.018745,-0.033654,...,-0.003616,-0.005935,0.000000,-0.014798,0.014000,0.000000,-0.014151,-0.010152,0.001626,-0.016970
2024-10-10,0.000981,-0.003172,0.008019,0.006269,-0.000557,0.002845,0.025843,0.003276,0.014950,-0.004975,...,0.008428,0.000000,-0.008696,0.060079,0.000000,-0.004348,0.000000,0.000000,0.006494,0.000000
2024-10-11,-0.007843,0.005702,0.016281,0.005742,-0.009849,-0.001527,-0.022979,-0.004082,-0.001637,-0.015000,...,-0.002322,-0.002985,-0.013158,-0.032066,-0.005917,-0.004367,0.000000,-0.002564,-0.011290,-0.004932


In [7]:
model_long_only = MeanRisk(
    risk_measure=RiskMeasure.VARIANCE,
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    min_weights=0.0,
    max_weights=0.3,
    portfolio_params=dict(name="Long-Only Max Sharpe"),
)

model_long_only.fit(X_train)

pred_long_only = model_long_only.predict(X_test)

c:\projects\adjusted_close\venv\Lib\site-packages\skfolio\moments\covariance\_base.py:362: UserWarning: The covariance matrix is not positive definite. The Clipping algorithm will be used to find the nearest positive definite covariance.
  covariance = cov_nearest(


In [8]:
# Если оставить бюджет нулевым на нашей истории портфель будет просто пустым
model_market_neutral = MeanRisk(
    risk_measure=RiskMeasure.CVAR,  
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    min_weights=-0.2, 
    max_weights=0.2,    
    budget = 1,
    portfolio_params=dict(name="Market Neutral Min CVaR")
)

model_market_neutral.fit(X_train)
pred_market_neutral = model_market_neutral.predict(X_test)

c:\projects\adjusted_close\venv\Lib\site-packages\skfolio\moments\covariance\_base.py:362: UserWarning: The covariance matrix is not positive definite. The Clipping algorithm will be used to find the nearest positive definite covariance.
  covariance = cov_nearest(


In [9]:
population = Population([pred_long_only, pred_market_neutral])

population.summary()

fig = population.plot_cumulative_returns()
fig.show(renderer="browser")
population.plot_composition()

Для ETF

In [10]:
Y = load_returns("data/etf")

print(Y.head())
Y_train, Y_test = train_test_split(Y, test_size=0.33, shuffle=False)


                 XLB       XLC       XLE       XLF       XLI       XLK  \
date                                                                     
2018-06-20 -0.003247  0.012411  0.004415 -0.002557  0.000682  0.002100   
2018-06-21 -0.010636 -0.006129 -0.018517 -0.002935 -0.012554 -0.007684   
2018-06-22  0.014564  0.004377  0.019952 -0.004779  0.003455 -0.003238   
2018-06-25 -0.015552 -0.020599 -0.020092 -0.010713 -0.012671 -0.020761   
2018-06-26  0.003819  0.001658  0.012628 -0.003359  0.003767  0.004038   

                 XLP      XLRE       XLU       XLV       XLY  
date                                                          
2018-06-20  0.000981  0.010794  0.000798  0.002121  0.004742  
2018-06-21  0.001955  0.005967  0.003387 -0.005762 -0.007124  
2018-06-22  0.008203  0.008740  0.006945  0.004495 -0.001704  
2018-06-25  0.005036 -0.002476  0.016549 -0.009184 -0.021738  
2018-06-26 -0.004241  0.005273  0.001164 -0.003090  0.007161  


In [11]:
model_long_only = MeanRisk(
    risk_measure=RiskMeasure.VARIANCE,
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    min_weights=0.0,
    max_weights=0.3,
    portfolio_params=dict(name="Long-Only Max Sharpe"),
)

model_long_only.fit(Y_train)
pred_long_only = model_long_only.predict(Y_test)



In [12]:

model_market_neutral = MeanRisk(
    risk_measure=RiskMeasure.CVAR,  
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    min_weights=-0.2, 
    max_weights=0.2,   
    budget = 0,
    portfolio_params=dict(name="Market Neutral Min CVaR"),
)

model_market_neutral.fit(Y_train)
pred_market_neutral = model_market_neutral.predict(Y_test)

In [13]:
population = Population([pred_long_only, pred_market_neutral])

population.summary()

fig = population.plot_cumulative_returns()
fig.show(renderer="browser")
population.plot_composition()

+ сравнение MaxSharpe, RiskParity и BlackLitterman

In [ ]:
recent_returns = Y_train.iloc[-800:].mean()
recent_returns


XLB     0.000397
XLC     0.000187
XLE     0.001467
XLF     0.000484
XLI     0.000423
XLK     0.000553
XLP     0.000191
XLRE    0.000109
XLU     0.000221
XLV     0.000291
XLY     0.000192
dtype: float64

In [32]:
print(recent_returns.index)
views = []
for i in range(11):
    print(str(recent_returns.index[i]) + ' == ' + str(recent_returns.iloc[i]))
    views.append(str(recent_returns.index[i]) + ' == ' + str(recent_returns.iloc[i]))
print(views)

Index(['XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV',
       'XLY'],
      dtype='str')
XLB == 0.0003971000076096444
XLC == 0.0001870733861541382
XLE == 0.0014666829035222583
XLF == 0.0004839808827312196
XLI == 0.00042330806468995896
XLK == 0.0005531063392079739
XLP == 0.000190805740866547
XLRE == 0.00010924843718733951
XLU == 0.00022140386874821685
XLV == 0.0002910663487403595
XLY == 0.00019202098370563983
['XLB == 0.0003971000076096444', 'XLC == 0.0001870733861541382', 'XLE == 0.0014666829035222583', 'XLF == 0.0004839808827312196', 'XLI == 0.00042330806468995896', 'XLK == 0.0005531063392079739', 'XLP == 0.000190805740866547', 'XLRE == 0.00010924843718733951', 'XLU == 0.00022140386874821685', 'XLV == 0.0002910663487403595', 'XLY == 0.00019202098370563983']


In [33]:
MaxSharpe = MeanRisk(
    risk_measure=RiskMeasure.VARIANCE,
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    min_weights=0,
    max_weights=0.2,
    budget = 1,
    portfolio_params=dict(name="Max Sharpe"),
)

MaxSharpe.fit(Y_train)
pred_MaxSharpe = MaxSharpe.predict(Y_test)

RiskParity = RiskBudgeting(
    risk_measure=RiskMeasure.VARIANCE,
    portfolio_params=dict(name="Risk Parity"),
)

RiskParity.fit(Y_train)
pred_RiskParity = RiskParity.predict(Y_test)

bl_prior = BL(
    views=views,
    tau=0.05
)

BlackLittermanP = MeanRisk(
    risk_measure=RiskMeasure.VARIANCE,
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    prior_estimator=bl_prior,
    min_weights=0.0,
    max_weights=0.3,
    portfolio_params=dict(name="Black-Litterman Max Sharpe"),
)

BlackLittermanP.fit(Y_train)
pred_BlackLitterman = BlackLittermanP.predict(Y_test)


In [34]:
population = Population([pred_MaxSharpe, pred_RiskParity,pred_BlackLitterman])

population.summary()

fig = population.plot_cumulative_returns()
fig.show(renderer="browser")
population.plot_composition()